In [1]:
from molsim import MolecularDynamics
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

# Introduction to Nose-Hoover Thermostats

Molecular Dynamics (MD) simulations are a pivotal tool in understanding the microscopic behavior of systems in statistical mechanics. To accurately represent thermodynamic ensembles, particularly the canonical or NVT (constant Number of particles, Volume, and Temperature) ensemble, it is essential to control the temperature of the system. The Nose-Hoover thermostat is a widely adopted method for sampling the NVT ensemble in MD simulations, offering a deterministic approach to temperature regulation.

## Sampling the NVT Ensemble in MD

In MD simulations, maintaining the desired temperature is crucial for accurately sampling the canonical ensemble. Traditional MD simulations naturally conserve the total energy, thus sampling the microcanonical (NVE) ensemble. To transition to the NVT ensemble, where temperature fluctuations are allowed while keeping the number of particles and volume constant, a thermostat mechanism is introduced. The Nose-Hoover thermostat achieves this by extending the system with additional degrees of freedom that act as a heat bath, enabling energy exchange between the system and the bath to regulate temperature.

## The Extended System Hamiltonian

The Nose-Hoover approach augments the original Hamiltonian of the system by incorporating an auxiliary degree of freedom, typically modeled as a harmonic oscillator, which serves as the heat bath. The extended Hamiltonian $\mathcal{H}_{\text{NH}}$ is given by:

\begin{equation}
\mathcal{H}_{\text{NH}} = \sum_{i=1}^{N} \frac{p_i^2}{2m_i} + U(q_i) + \frac{p_\zeta^2}{2Q} + g k_B T \ln \zeta
\end{equation}

where:
- $p_i$ and $q_i$ are the momenta and positions of the $i$-th particle.
- $m_i$ is the mass of the $i$-th particle.
- $U(q_i)$ is the potential energy of the system.
- $\zeta$ is the additional dynamical variable representing the thermostat.
- $p_\zeta$ is the momentum conjugate to $\zeta$.
- $Q$ is the Nose-Hoover mass parameter, controlling the coupling strength to the heat bath.
- $g$ is the number of degrees of freedom in the system.
- $k_B$ is the Boltzmann constant, and $T$ is the target temperature.

## The New Equations of Motion

Deriving the equations of motion from the extended Hamiltonian using Hamilton's equations yields the following set of coupled differential equations:

\begin{equation}
\begin{aligned}
\dot{q}_i &= \frac{\partial \mathcal{H}_{\text{NH}}}{\partial p_i} = \frac{p_i}{m_i} \\
\dot{p}_i &= -\frac{\partial \mathcal{H}_{\text{NH}}}{\partial q_i} - \zeta p_i \\
\dot{\zeta} &= \frac{\partial \mathcal{H}_{\text{NH}}}{\partial p_\zeta} = \frac{p_\zeta}{Q} \\
\dot{p}_\zeta &= -\frac{\partial \mathcal{H}_{\text{NH}}}{\partial \zeta} = g k_B T - \sum_{i=1}^{N} \frac{p_i^2}{m_i}
\end{aligned}
\end{equation}

These equations ensure that the system exchanges energy with the thermostat, thereby regulating the temperature. The variable $\zeta$ acts as a friction coefficient that adjusts dynamically to maintain the desired kinetic energy corresponding to the target temperature.

## Intricacies of the Timescale Parameter $ Q $

The parameter $ Q $, often referred to as the Nose-Hoover mass, plays a critical role in the performance of the thermostat. It determines the timescale over which the thermostat responds to deviations in temperature:

- **Small $ Q $**: The thermostat reacts quickly, tightly coupling the system to the heat bath. While this can efficiently control temperature, it may interfere with the natural dynamics of the system, potentially distorting physical properties.

- **Large $ Q $**: The thermostat responds more slowly, minimally perturbing the system's intrinsic dynamics. However, if $Q$ is too large, the thermostat may not effectively regulate the temperature, leading to insufficient sampling of the NVT ensemble.

Choosing an appropriate $ Q $ is thus a balance between accurate temperature control and the preservation of the system's natural dynamical behavior. In practice, $Q$ is often selected based on empirical testing or guidelines related to the characteristic timescales of the system being simulated.

Here, we initialize the timescale parameter for the system as $Q = g \tau^2 \Delta t$, for timescale $\tau$. Doing so, the timescale parameter can be set to an amount of MD steps that determines the frequency of the harmonic oscillator. A very large $\tau$ gives long frequency oscillations, where as a smaller $\tau$ gives short frequency oscillations. 



<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

# Question 1
Study the thermostat code in `molsim/molecularDynamics/thermostats.cpp`. Here, you can see the algorithm used to rescale the velocities towards a certain temperature by the heat bath. In the function `NoseHooverNVT::getEnergy`, the Nose-Hoover part of the Hamiltonian is computed to check for conservation of the Hamiltonian. Run a simulation using the `useNoseHoover` flag set to True and the `noseHooverTimeScaleParameter` set to an integer value to run an NVT simulation. 

Plot the conserved energy (`md.conservedEnergies`), the potential energy (`md.potentialEnergies`), the kinetic energy (`md.kineticEnergies`) and the thermostat energy and total system energy (derived from previous series). 

**Hint**: to add kinetic and potential energy, first convert to numpy arrays such that you can do array math.

In [ ]:
# Run a molecular dynamics simulation
# start refactor
temperature = None
dt = None
useNoseHoover = None
noseHooverTimeScaleParameter = None
# end refactor


md = MolecularDynamics(
    numberOfParticles=200,
    temperature=temperature,
    dt=dt,
    boxSize=8.0,
    numberOfEquilibrationSteps=int(1e4),
    numberOfProductionSteps=int(1e5),
    outputPDB=True,
    logLevel=0,
    seed=12,
    sampleFrequency=100,
    useNoseHoover=useNoseHoover,
    noseHooverTimeScaleParameter=noseHooverTimeScaleParameter,
)
md.run()

In [ ]:
# Plot the results
fig, ax = plt.subplots()

kin = np.array(md.kineticEnergies)
pot = np.array(md.potentialEnergies)
cons = np.array(md.conservedEnergies)

# start refactor
ax.plot([], [], label=r"$E_{kin}$")
ax.plot([], [], label=r"$E_{pot}$")
ax.plot([], [], label=r"$E_{total}$")
ax.plot([], [], label=r"$E_{Conserved}$")
ax.plot([], [], label=r"$E_{NH}$")
# end refactor
ax.legend()
ax.set_xlabel("Time")
ax.set_ylabel("Energy")

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

# Question 2
A crucial validation for any thermostat is verifying that the kinetic energy of the system's particles adheres to the Maxwell-Boltzmann distribution corresponding to the target temperature $T$. This check involves analyzing the velocity - or temperature - distribution of particles to ensure it matches the expected statistical distribution. By confirming that the Nose-Hoover thermostat correctly samples the Maxwell-Boltzmann distribution, one can be confident that the temperature control is accurate and that the simulation represents the canonical ensemble.

Here we will run a simulation in both NVE and NVT and validate that the NVT simulation correctly represents the expected Maxwell-Boltzmann distribution. Calculate the variance in the temperature based on

\begin{equation}
\sigma_T^2 = \frac{2T^2}{g}
\end{equation}


In [ ]:
# start refactor
numberOfParticles = None
temperature = None
density = None
# end refactor

md_NVE = MolecularDynamics(
    numberOfParticles=numberOfParticles,
    temperature=temperature,
    dt=0.005,
    boxSize=np.cbrt(numberOfParticles / density),
    numberOfEquilibrationSteps=int(1e5),
    numberOfProductionSteps=int(1e5),
    outputPDB=False,
    seed=12,
    sampleFrequency=100,
    useNoseHoover=False,
)
md_NVE.run()

md_NVT = MolecularDynamics(
    numberOfParticles=numberOfParticles,
    temperature=temperature,
    dt=0.005,
    boxSize=np.cbrt(numberOfParticles / density),
    numberOfEquilibrationSteps=int(1e5),
    numberOfProductionSteps=int(1e5),
    outputPDB=False,
    seed=12,
    sampleFrequency=100,
    useNoseHoover=True,
    noseHooverTimeScaleParameter=100,
)
md_NVT.run()

In [ ]:
fig, ax = plt.subplots()

# plot the observed temperatures from simulation
ax.hist(md_NVE.observedTemperatures, bins=50, density=True, edgecolor="black", alpha=0.7, label="NVE")
ax.hist(md_NVT.observedTemperatures, bins=50, density=True, edgecolor="black", alpha=0.7, label="NVT")

# calculate the degrees of freedom and variance in temperature
degreesOfFreedom = 3 * numberOfParticles - 3

# start refactor
maxwellBoltzmannVariance = None
# end refactor


# Compute the Maxwell-Boltzmann distribution
x = np.linspace(np.min(md_NVT.observedTemperatures), np.max(md_NVT.observedTemperatures), 1000)
y = norm.pdf(x, loc=temperature, scale=np.sqrt(maxwellBoltzmannVariance))

ax.plot(x, y, label="Maxwell-Boltzmann", c="red")
ax.set_xlabel(r"Temperature, $T$ / $\varepsilon$")
ax.set_ylabel("Density")
ax.legend()

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

# Question 3
The timescale parameter $ Q $ influences how the thermostat responds to temperature fluctuations. To understand its impact, it is essential to analyze the temporal behavior of the system's temperature for various $ Q $ values. This involves generating and examining temperature time series data to observe how quickly and effectively the thermostat stabilizes temperature deviations. By comparing fluctuations across different $ Q $ settings, one can determine the optimal parameter that balances responsive temperature control with minimal disturbance to the system's natural dynamics.

Analyze the fluctuation for a different range of values of $\tau$ and see what happens to the temperature fluctuations at low and high values of $\tau$. What effect do you see at low values of $\tau$ and how does this affect the sampling of the canonical ensemble? And for high values of $\tau$? What is a reasonable value of $\tau$ for this system?

In [ ]:
temperature = 1.0
dt = 5e-3

# start refactor
taus = [10]
# end refactor

fig, ax = plt.subplots(1, len(taus), figsize=(3 * len(taus), 4), sharey=True)

for i, tau in enumerate(taus):
    md_tau = MolecularDynamics(
        numberOfParticles=100,
        temperature=temperature,
        dt=dt,
        boxSize=10.0,
        numberOfEquilibrationSteps=int(1e4),
        numberOfProductionSteps=int(1e5),
        outputPDB=True,
        logLevel=0,
        seed=12,
        sampleFrequency=100,
        useNoseHoover=True,
        noseHooverTimeScaleParameter=tau,
    )
    md_tau.run()
    ax[i].plot(md_tau.time, md_tau.observedTemperatures, lw=1, label=r"$\tau$= " + str(tau), c="black")
    ax[i].set_title(r"$\tau=$" + str(tau) + r"$\Delta t$")

    ax[i].set_ylim(0.5 * temperature, 1.5 * temperature)
    ax[i].set_xlabel("Time")
ax[0].set_ylabel(r"Temperature, $T$ / $\varepsilon$")
fig.tight_layout()